# **Deep Residual MLP**

## **1. Import Libraries**

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## **2. Load Dataset**

In [ ]:
# Load dataset and drop unrelated identifiers
df = pd.read_csv("dataset\\digital_marketing_campaign_dataset.csv").drop(
    columns=["CustomerID", "AdvertisingPlatform", "AdvertisingTool"]
)
df.head()

## **3. Data Preprocessing**

Based on inferential analysis results, only statistically significant variables (p-value < 0.05) are retained. `ConversionRate` is excluded despite significance to **prevent data leakage**.

In [ ]:
# Selected features from inferential analysis (excluding ConversionRate - data leakage risk)
FEATURES = [
    "CampaignType",       # Chi-square: significant
    "AdSpend",            # Mann-Whitney U: significant
    "ClickThroughRate",   # Mann-Whitney U: significant
    "WebsiteVisits",      # Mann-Whitney U: significant
    "PagesPerVisit",      # Mann-Whitney U: significant
    "TimeOnSite",         # Mann-Whitney U: significant
    "EmailOpens",         # Mann-Whitney U: significant
    "EmailClicks",        # Mann-Whitney U: significant
    "PreviousPurchases",  # Mann-Whitney U: significant
    "LoyaltyPoints"       # Mann-Whitney U: significant
]
TARGET = "Conversion"

# Encode categorical variable CampaignType
le = LabelEncoder()
df["CampaignType"] = le.fit_transform(df["CampaignType"])

X = df[FEATURES].values
y = df[TARGET].values

# Train / Validation / Test split: 70% / 15% / 15%
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

# Standardise numerical features (fit only on training data)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# Convert to PyTorch tensors
def to_tensors(X, y):
    return (
        torch.tensor(X, dtype=torch.float32).to(device),
        torch.tensor(y, dtype=torch.float32).to(device)
    )

X_train_t, y_train_t = to_tensors(X_train, y_train)
X_val_t,   y_val_t   = to_tensors(X_val,   y_val)
X_test_t,  y_test_t  = to_tensors(X_test,  y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=128, shuffle=True)

print(f"Train : {X_train.shape[0]} samples")
print(f"Val   : {X_val.shape[0]} samples")
print(f"Test  : {X_test.shape[0]} samples")
print(f"Features: {len(FEATURES)}")

## **4. Model Architecture — Deep Residual MLP**

A **Residual MLP** adapts the skip-connection idea from ResNet to tabular data. Each residual block learns a **residual function** $F(x)$ instead of directly mapping the input:

$$\text{output} = F(x) + x = \text{Linear} \rightarrow \text{BN} \rightarrow \text{ReLU} \rightarrow \text{Linear} \rightarrow \text{BN} + x$$

This allows **gradient to flow directly** through skip connections during backpropagation, enabling a deeper network without vanishing gradient issues — a common problem for plain deep MLPs on tabular data.

**Architecture summary:**
- Input projection layer: 10 → 128 neurons
- 4 × Residual blocks (128 → 128): each block has 2 linear layers + Batch Normalization + ReLU + Dropout (0.3)
- Output head: 128 → 64 → 1 with Sigmoid activation

In [ ]:
class ResidualBlock(nn.Module):
    """Single residual block: two linear layers with skip connection."""
    def __init__(self, dim: int, dropout: float = 0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.block(x) + x)   # skip connection


class DeepResidualMLP(nn.Module):
    """Deep Residual MLP for binary classification on tabular data."""
    def __init__(self, input_dim: int, hidden_dim: int = 128,
                 num_blocks: int = 4, dropout: float = 0.3):
        super().__init__()

        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )

        # Stack of residual blocks
        self.res_blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim, dropout) for _ in range(num_blocks)]
        )

        # Output head
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.res_blocks(x)
        return self.output_head(x).squeeze(1)


INPUT_DIM  = X_train.shape[1]   # 10
model = DeepResidualMLP(input_dim=INPUT_DIM).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")

## **5. Training**

**Configuration:**
- **Loss function**: Binary Cross-Entropy (`BCELoss`) — standard for binary classification
- **Optimiser**: Adam with learning rate `1e-3` and weight decay `1e-4` (L2 regularisation)
- **Scheduler**: ReduceLROnPlateau — reduces LR by factor 0.5 if validation loss stagnates for 5 epochs
- **Early stopping**: patience of 15 epochs to prevent overfitting

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

EPOCHS        = 150
PATIENCE      = 15

best_val_loss = float('inf')
patience_counter = 0
best_state    = None

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    train_loss, train_correct = 0.0, 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item() * len(y_batch)
        train_correct += ((preds >= 0.5) == y_batch.bool()).sum().item()

    train_loss /= len(y_train)
    train_acc   = train_correct / len(y_train)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t)
        val_loss  = criterion(val_preds, y_val_t).item()
        val_acc   = ((val_preds >= 0.5) == y_val_t.bool()).float().mean().item()

    scheduler.step(val_loss)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    # ── Early stopping ────────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

# Restore best weights
model.load_state_dict(best_state)
print("\nTraining complete. Best model weights restored.")

## **6. Learning Curves**

In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Deep Residual MLP — Learning Curves", fontsize=14, fontweight="bold")

# Loss
axes[0].plot(epochs_ran, history["train_loss"], label="Train Loss", linewidth=2)
axes[0].plot(epochs_ran, history["val_loss"],   label="Val Loss",   linewidth=2, linestyle="--")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary Cross-Entropy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_ran, history["train_acc"], label="Train Accuracy", linewidth=2)
axes[1].plot(epochs_ran, history["val_acc"],   label="Val Accuracy",   linewidth=2, linestyle="--")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## **7. Evaluation on Test Set**

In [ ]:
model.eval()
with torch.no_grad():
    y_prob = model(X_test_t).cpu().numpy()

y_pred = (y_prob >= 0.5).astype(int)

acc     = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

summary = pd.DataFrame({
    "Metric" : ["Accuracy", "ROC-AUC"],
    "Score"  : [round(acc, 4), round(roc_auc, 4)]
})
summary

### **7.1 Classification Report**

In [ ]:
report = classification_report(y_test, y_pred, target_names=["Not Converted (0)", "Converted (1)"],
                                output_dict=True)
pd.DataFrame(report).T.round(4)

### **7.2 Confusion Matrix**

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Not Converted", "Converted"],
    yticklabels=["Not Converted", "Converted"]
)
plt.title("Confusion Matrix — Deep Residual MLP", fontweight="bold")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

### **7.3 ROC Curve**

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random Classifier")
plt.title("ROC Curve — Deep Residual MLP", fontweight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## **8. Conclusion**

The Deep Residual MLP uses **skip connections** to stabilise gradient flow across 4 stacked residual blocks, enabling the network to learn complex non-linear patterns in the marketing data without suffering from vanishing gradients — a key limitation of plain deep MLPs.

**Key design decisions and their justifications:**
- **Residual connections**: allow the model to selectively learn residuals, avoiding degradation in deeper layers.
- **Batch Normalisation**: reduces internal covariate shift, stabilising and accelerating training.
- **Dropout (0.3)**: acts as a regulariser to reduce overfitting on tabular data.
- **ReduceLROnPlateau + Early Stopping**: prevents overfitting and unnecessary computation.
- **Feature selection from inferential analysis**: only the 10 statistically significant variables (p < 0.05) are used, improving signal-to-noise ratio.

Based on the evaluation results, the model demonstrates strong predictive performance on conversion classification using purely behavioural and campaign-level features.